In [1]:
# Cell 1 - Imports
import requests
import pandas as pd
import time


In [2]:
# Cell 2 — Configuration (only this cell needs to be updated)
BASE_URL = "https://findata-qa.worldbank.org:9047"
TOKEN    = "3/Cw8MfRR5CgJxP9cfVlot3cd95gGBfZvwEa7jp3tBOU9UWfFCJ8e6k6KAhF0w=="
SCHEMA   = ["publishFINDATAREP", "IFCTRE", "Application", "Murex"]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# ── All your queries ──
QUERIES = {
    "perf_attribution"      : 'SELECT * FROM "Perf_Attribution"',
    "perf_benchmark"        : 'SELECT * FROM "Perf_Benchmark_Weights"',
    "perf_vs_funding"       : 'SELECT * FROM "Perf_Vs_Funding_Snap"',
    "plvar_snapshot"        : 'SELECT * FROM "Plvar_Snapshot"',
    "portfolio_details"     : 'SELECT * FROM "Portfolio_Details"',
    "securities_detail"     : 'SELECT * FROM "Securities_Detail"',
    "trade_mktval_detail"   : 'SELECT * FROM "Trade_Mktval_Detail"',
    "security_prices_detail": 'SELECT * FROM "Security_Prices_Detail"',
    "inv_pnl_summary"       : 'SELECT * FROM "Inv_Pnl_Summary"',
    "inv_security_position" : 'SELECT * FROM "Inv_Security_Position"',  # ← new
}

# ── Only change this line to switch query ──
SQL_QUERY = QUERIES[ "inv_pnl_summary" ]


In [3]:
# Cell 3 — Display function
def display_df(df):
    return df.style.set_properties(**{
        'white-space': 'nowrap',
        'font-size': '11px',
        'text-align': 'left',
        'border': '1px solid lightgrey',
        'color': 'black'
    }).set_table_styles([
        {
            'selector': 'thead th',
            'props': [
                ('white-space', 'nowrap'),
                ('font-size', '11px'),
                ('font-weight', 'bold'),
                ('background-color', '#4472C4'),
                ('color', 'white'),
                ('text-align', 'center'),
                ('border', '1px solid lightgrey')
            ]
        },
        {
            'selector': 'tbody tr:nth-child(even)',
            'props': [('background-color', '#f2f2f2'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:nth-child(odd)',
            'props': [('background-color', 'white'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:hover',
            'props': [('background-color', '#d6e4f0'), ('color', 'black')]
        }
    ])


In [4]:
# Cell 4 — Submit the SQL job
response = requests.post(
    f"{BASE_URL}/api/v3/sql",
    headers=HEADERS,
    json={
        "sql": SQL_QUERY,
        "context": SCHEMA
    },
    verify=True,
    timeout=30
)
response.raise_for_status()
job_id = response.json().get("id")
print(f"Job submitted successfully.")
print(f"Job ID: {job_id}")


Job submitted successfully.
Job ID: 1598a733-0b52-e7cd-1eac-64eb857a4800


In [5]:
# Cell 5 — Poll until job completes
url = f"{BASE_URL}/api/v3/job/{job_id}"
for attempt in range(30):
    response = requests.get(url, headers=HEADERS, verify=True)
    response.raise_for_status()
    status = response.json().get("jobState")
    print(f"Attempt {attempt + 1}: {status}")
    if status == "COMPLETED":
        print("Job completed!")
        break
    elif status in ("FAILED", "CANCELED"):
        print(f"Job failed: {response.json().get('errorMessage')}")
        break
    time.sleep(2)


Attempt 1: COMPLETED
Job completed!


In [6]:
# Cell 6 — Fetch results and build DataFrame
response = requests.get(
    f"{BASE_URL}/api/v3/job/{job_id}/results?offset=0&limit=500",
    headers=HEADERS,
    verify=True
)
response.raise_for_status()
data = response.json()

columns   = [col["name"] for col in data.get("schema", [])]
all_rows  = data.get("rows", [])
total     = data.get("rowCount", None)

# ── Paginate if more than 500 rows ──
offset = 500
while True:
    r = requests.get(
        f"{BASE_URL}/api/v3/job/{job_id}/results?offset={offset}&limit=500",
        headers=HEADERS,
        verify=True
    )
    batch = r.json().get("rows", [])
    if not batch:
        break
    all_rows.extend(batch)
    offset += len(batch)
    print(f"Fetched {offset} rows so far...")

df = pd.DataFrame(all_rows, columns=columns)
print(f"Total Rows    : {len(df)}")
print(f"Total Columns : {len(df.columns)}")


Fetched 1000 rows so far...
Fetched 1500 rows so far...
Fetched 2000 rows so far...
Fetched 2500 rows so far...
Fetched 3000 rows so far...
Fetched 3500 rows so far...
Fetched 4000 rows so far...
Fetched 4500 rows so far...
Fetched 5000 rows so far...
Fetched 5500 rows so far...
Fetched 6000 rows so far...
Fetched 6500 rows so far...
Fetched 7000 rows so far...
Fetched 7500 rows so far...
Fetched 8000 rows so far...
Fetched 8500 rows so far...
Fetched 9000 rows so far...
Fetched 9500 rows so far...
Fetched 10000 rows so far...
Fetched 10500 rows so far...
Fetched 11000 rows so far...
Fetched 11500 rows so far...
Fetched 12000 rows so far...
Fetched 12500 rows so far...
Fetched 13000 rows so far...
Fetched 13456 rows so far...
Total Rows    : 13456
Total Columns : 49


In [15]:
display_df(df.head(10))   # first 10 rows

,Business_Date,MX_REF_DATA,MX_REF_JOB,Cash,Cash_USD,MX_CD_ELIG,MX_CD_INC_PTF,MX_CD_INC_USD,Contract_Origin,Root_Contract_ID,Country,CTP_Code,Currency,MX_DATA_DATE,Data_Label,MX_DEAL_TYPE,DTD_pnl,DTD_pnl_usd,Data_Generation_Date,Data_Generation_Desk,Data_Generation_Group,Data_Generation_Time,Data_Generation_User,Instrument,Level1_sector,Level2_sector,MX_L3_PORTFO,MX_L4_PORTFO,Package_ID,LTD_pnl,LTD_pnl_usd,MTD_Pnl,MTD_Pnl_USD,Market_Value,Market_Value_USD,Trade_ID,Book_ID,MX_PTF_CCY,Trade_Type_Desc,Tranche_ID,Trade_Typology,YTD_pnl,YTD_pnl_USD,UUID,FeedName,ProcessedTimeStamp,AccountingDate,BatchID,LastProcessedBatchId
0,2026-07-17,19418,979472,117776.899167,117776.899167,N,0.000000,0.000000,1696574,1696574,FR,BSUIFR,USD,2026-07-17,EOD,Asset,90123.968434,90123.970000,2026-07-19,FOD EOD,FO_EOD,45667,None,USD SOFR A 1Y,None,None,LAM_STRATEGIC,LAM_STG_GRE,IFC KR310101GEA8,90123.968434,90123.970000,90123.968434,90123.970000,-27652.930733,-27652.930733,1697180,None,USD,Interest rate swaps,KR_SOV,IRS,90123.968434,90123.970000,9f0906b6-f896-404c-bcc5-a449bd1ffdd7-20260719124959,FINVPFPLSUMSE,2026-07-19 12:45:39.000,2026-07-19,202607170001,202607170001
1,2026-07-17,19418,979472,0.000000,0.000000,N,0.000000,0.000000,1696630,1696630,CA,ROYCCD,USD,2026-07-17,EOD,Asset,211271.427797,211271.430000,2026-07-19,FOD EOD,FO_EOD,45667,None,USD SOFR A 1Y,None,None,LAM_STRATEGIC,LAM_STG_SP_USA,None,211271.427797,211271.430000,211271.427797,211271.430000,211271.427797,211271.427797,1697236,None,USD,Interest rate swaps,None,IRS,211271.427797,211271.430000,518ce5d8-0be5-48f6-a6b7-2fd60bdf667d-20260719124959,FINVPFPLSUMSE,2026-07-19 12:45:39.000,2026-07-19,202607170001,202607170001
2,2026-07-17,19418,979472,0.000000,0.000000,N,0.000000,0.000000,1696639,1696639,CA,CIBCCD,USD,2026-07-17,EOD,Asset,216418.960423,216418.960000,2026-07-19,FOD EOD,FO_EOD,45667,None,USD SOFR A 1Y,None,None,LAM_STRATEGIC,LAM_STG_GRE,IFC US91282CQT17,216418.960423,216418.960000,216418.960423,216418.960000,216418.960423,216418.960423,1697245,None,USD,Interest rate swaps,USD_ROLLDOWN,IRS,216418.960423,216418.960000,e0bd8367-c0a6-4420-9da7-148c99f9b0ed-20260719124959,FINVPFPLSUMSE,2026-07-19 12:45:39.000,2026-07-19,202607170001,202607170001
3,2026-07-17,19418,979472,997869.027171,997869.027171,N,0.000000,0.000000,1696640,1696640,SG,DBSZSG,USD,2026-07-17,EOD,Asset,730659.943814,730659.940000,2026-07-19,FOD EOD,FO_EOD,45667,None,SGD-USD SOFR S-3M,None,None,LAM_STRATEGIC,LAM_STG_GRE,IFC SG7CF3000005,730659.943814,730659.940000,730659.943814,730659.940000,-267209.083358,-267209.083358,1697246,None,USD,Currency swaps,SINGAPORE,Xccy Swap,730659.943814,730659.940000,ce8362fa-db1b-457d-9de2-d4de761da7b5-20260719124959,FINVPFPLSUMSE,2026-07-19 12:45:39.000,2026-07-19,202607170001,202607170001
4,2026-07-17,19418,979472,53842.338412,53842.338412,N,0.000000,0.000000,1696647,1696647,US,FUNBUS,AUD,2026-07-17,EOD,Asset,38795.381968,38795.380000,2026-07-19,FOD EOD,FO_EOD,45667,None,AUD BBSW S 3M,None,None,LAM_STRATEGIC,LAM_STG_GRE,IFC AU3CB0295749,38795.381968,38795.380000,38795.381968,38795.380000,-15046.956444,-15046.956444,1697253,None,USD,Interest rate swaps,AUD_CARRY,IRS,38795.381968,38795.380000,1602edd3-46b4-46bb-974c-39fd21d5baa7-20260719124959,FINVPFPLSUMSE,2026-07-19 12:45:39.000,2026-07-19,202607170001,202607170001
5,2026-07-17,19418,979472,-304872.301325,-304872.301325,N,0.000000,0.000000,1696727,1696727,CA,ROYCCD,CAD,2026-07-17,EOD,Asset,-1434648.969067,-1434648.970000,2026-07-19,FOD EOD,FO_EOD,45667,None,CAD CORRA S 6M,None,None,LAM_STRATEGIC,LAM_STG_GRE,IFC CAC23264AQ42,-1434648.969067,-1434648.970000,-1434648.969067,-1434648.970000,-1129776.667742,-1129776.667742,1697333,None,USD,Interest rate swaps,CAD_ROLLDOWN,IRS,-1434648.969067,-1434648.970000,426294e4-df8f-473c-a957-3174f23eefe0-20260719124959,FINVPFPLSUMSE,2026-07-19 12:45:39.000,2026-07-19,202607170001,202607170001
6,2026-07-17,19418,979472,0.000000,0.000000,N,0.000000,0.000000,1696732,1696732,CA,BOFMCA,USD,2026-07-17,EOD,As

In [8]:
display_df(df.tail(10))   # last 10 rows

,Business_Date,MX_REF_DATA,MX_REF_JOB,CASH,CASH_T1,CASH_T1_US,CASH_USD,MX_DATA_DATE,Data_Label,DEAL_TYPE,MX_FTP,Data_Generation_Date,Data_Generation_Desk,Data_Generation_Group,Data_Generation_Time,Data_Generation_User,MX_L3_LQDTY,Portfolio_Level_3,MX_L3_STRTGY,Portfolio_Level_4,MX_L4_TRACKR,MX_MV,MV_CASH,MV_CASH_usd,MX_MV_CASH_Y1,MV_T1,MV_T1_USD,MV_usd,MX_PL_YTD,MX_PL_YTD_US,MX_POOL_FTP,Portfolio_Currency,Strategy,MX_TRADE_NUM,MX_TYPOLOGY,UUID,FeedName,ProcessedTimeStamp,AccountingDate,BatchID,LastProcessedBatchId
12211,2026-07-23,19606,986480,-527268850665.999939,-527268850665.999939,-164542258.000000,-165297730.708649,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,LCY_POOLS,0,POOL_LIQ_COP,0,556525546284.738525,29256695618.738564,9171915.594298,0.000000,556297898311.307007,173601213.000000,174469646.302947,29256695618.738564,9171915.594298,0,COP,None,1701482,Investment Bond,d888989a-2cc7-49b6-b730-a3a86b033570-20260724064957,FINVPRFVFUNSE,2026-07-24 06:49:27.000,2026-07-23,202607230001,202607230001
12212,2026-07-23,19606,986480,444.899909,443.852231,444.000000,444.899909,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,MF_MOPS_CASH,0,CASH_MF_MISSION,0,0.000000,444.899909,444.899909,0.000000,0.000000,0.000000,0.000000,444.899909,444.899909,0,USD,None,1716483,SCF,7610a813-4b3e-4620-8029-60d3afe7b475-20260724064957,FINVPRFVFUNSE,2026-07-24 06:49:27.000,2026-07-23,202607230001,202607230001
12213,2026-07-23,19606,986480,385727318.477385,387315016.779703,387315017.000000,385727318.477385,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,MF_MOPS_CASH,0,CASH_MF_MISSION,0,0.000000,385727318.477385,385727318.477385,0.000000,0.000000,0.000000,0.000000,385727318.477385,385727318.477385,0,USD,None,1716484,SCF,9288b0f7-c0e3-41fe-be2c-0ac20b0afaf0-20260724064957,FINVPRFVFUNSE,2026-07-24 06:49:27.000,2026-07-23,202607230001,202607230001
12214,2026-07-23,19606,986480,48291391.061282,48270356.105614,48270356.000000,48291391.061282,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,MF_MOPS_CASH,0,CASH_MF_MISSION,0,0.000000,48291391.061282,48291391.061282,0.000000,0.000000,0.000000,0.000000,48291391.061282,48291391.061282,0,USD,None,1716485,SCF,a7b65f16-24cf-4855-8f68-85f3b20cb82b-20260724064957,FINVPRFVFUNSE,2026-07-24 06:49:27.000,2026-07-23,202607230001,202607230001
12215,2026-07-23,19606,986480,28253.920266,28259.789866,28260.000000,28253.920266,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,MF_MOPS_CASH,0,CASH_MF_MISSION,0,0.000000,28253.920266,28253.920266,0.000000,0.000000,0.000000,0.000000,28253.920266,28253.920266,0,USD,None,1716486,SCF,6b888e67-53cc-48d4-9917-24fa71456959-20260724064957,FINVPRFVFUNSE,2026-07-24 06:49:27.000,2026-07-23,202607230001,202607230001
12216,2026-07-23,19606,986480,9998.063139,10002.903092,10003.000000,9998.063139,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,MF_MOPS_CASH,0,CASH_MF_MISSION,0,0.000000,9998.063139,9998.063139,0.000000,0.000000,0.000000,0.000000,9998.063139,9998.063139,0,USD,None,1716487,SCF,7b7e2e18-ccba-4c35-9d2d-d17e7168636f-20260724064957,FINVPRFVFUNSE,2026-07-24 06:49:27.000,2026-07-23,202607230001,202607230001
12217,2026-07-23,19606,986480,1423811682.680778,1432451271.836553,1432451272.000000,1423811682.680778,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,MF_MOPS_CASH,0,CASH_MF_MISSION,0,0.000000,1423811682.680778,1423811682.680778,0.000000,0.000000,0.000000,0.000000,1423811682.680778,1423811682.680778,0,USD,None,1716488,SCF,7db83538-0eff-4241-8e59-7f52834fd8e0-20260724064957,FINVPRFVFUNSE,2026-07-24 06:49:27.000,2026-07-23,202607230001,202607230001
12218,2026-07-23,19606,986480,51.910478,51.900974,52.000000,51.910478,2026-07-23,EOD,Asset,0,2026-07-24,FOD EOD,FO_EOD,21063,None,0,MF_MOPS_CASH,0,CASH_MF_MISSION,0,0.000000,51.910478,51.910478,0.000000,0.000000,0.000000,0.000000,51.910478,51.910478,0,USD,None,1716489,SCF,d5c79609-4854-4e57-a21b-fe6f7feb0f09-20260724064957,FINVPRFVFUNSE,2026-07-24 06

In [ ]:
# Cell 7 — Display the DataFrame
#display_df(df.head(10))   # first 10 rows
#display_df(df.tail(10))   # last 10 rows
display_df(df)            # everything



In [ ]:
df.tail()

In [ ]:
# Cell 8 — Optional: Save to CSV or Excel
df.to_csv("output.csv", index=False)
print("Saved to output.csv")

df.to_excel("output.xlsx", index=False)
print("Saved to output.xlsx")
